# Modeling tuning


## Setup

In [9]:
try:
    from synthefy_nori import NoriRegressor
    NORI_AVAILABLE = True
except ImportError:
    NORI_AVAILABLE = False

In [10]:
import logging
import os
import sys
import tempfile
from pathlib import Path
import pandas as pd
import xgboost as xgb
import yaml



import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

sys.path.insert(0, str(Path.cwd().parent / "scripts"))  # allow importing sibling scripts modules

from features import FEATURE_COLUMNS, prepare_datasets
from model import _recency_weights, walk_forward_evaluate
# from explain import feature_contributions
# from model import evaluate_models, predict_tomorrow, train_and_save_models, walk_forward_evaluate
# from model import DEFAULT_XGB_PARAMS
from strava_client import fetch_activities_dataframe, out_of_range_dates
from weather_client import DEFAULT_LAT, DEFAULT_LON, fetch_historical_weather, fetch_tomorrow_forecast

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
load_dotenv(Path.cwd().parent / ".env")  # repo root .env

True

In [11]:
USE_LIVE_DATA = False  # set True to pull real Strava activities + Open-Meteo forecast via .env secrets

if USE_LIVE_DATA:
    activities = fetch_activities_dataframe(days_back=365*2)
    weather = fetch_tomorrow_forecast()
    activity_dates = pd.to_datetime(activities['date'], errors='coerce').dt.date.dropna()
    historical_weather = (
        fetch_historical_weather(activity_dates.min(), activity_dates.max())
        if not activity_dates.empty
        else None
    )
    if historical_weather is not None:
        home_lat = float(os.getenv('FORECAST_LAT', DEFAULT_LAT))
        home_lon = float(os.getenv('FORECAST_LON', DEFAULT_LON))
        stale_dates = out_of_range_dates(activities, home_lat, home_lon)
        historical_weather = historical_weather.drop(index=list(stale_dates), errors='ignore')
    prepared = prepare_datasets(activities=activities, tomorrow_weather=weather, historical_weather=historical_weather)
    historical = prepared.historical
else:
    historical = pd.read_csv("local_data/historical.csv")



In [12]:
with open("../params.yaml", "r") as f:
    params = yaml.safe_load(f)

In [20]:
from itertools import product
from sklearn.metrics import roc_auc_score, accuracy_score

configured_features = params["model"].get("features", FEATURE_COLUMNS)
target = "will_train_tomorrow"
missing_required = {target, "as_of_date"} - set(historical.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")

features = [feature for feature in configured_features if feature in historical.columns]
missing_features = sorted(set(configured_features) - set(features))
if missing_features:
    if USE_LIVE_DATA:
        raise ValueError(f"Prepared data is missing configured model features: {missing_features}")
    print(f"Cached historical data lacks {missing_features}; tuning with available features only. Set USE_LIVE_DATA=True to refresh it.")
if not features:
    raise ValueError("No configured model features are available in the historical data")

tuning_data = historical.copy()
tuning_data["as_of_date"] = pd.to_datetime(tuning_data["as_of_date"], errors="coerce")
tuning_data = tuning_data.dropna(subset=["as_of_date", target]).sort_values("as_of_date")

split_idx = int(len(tuning_data) * params["model"].get("train_test_split_frac", 0.8))
clf_train, clf_valid = tuning_data.iloc[:split_idx], tuning_data.iloc[split_idx:]
if clf_train.empty or clf_valid.empty:
    raise ValueError("The chronological train/validation split is empty")

y_train = clf_train[target].astype(int)
y_valid = clf_valid[target].astype(int)
if y_train.nunique() < 2 or y_valid.nunique() < 2:
    raise ValueError("Both train and validation splits must contain both target classes")

configured = dict(params["model"]["xgboost"])

# Edit these candidate lists to explore a different grid. Every combination is fit once.
# Current grid: 3 depths x 3 learning rates x 2 tree counts x 3 half-lives = 54 fits.
search_space = {
    "max_depth": [3, 4, 5],
    "eta": [0.005,0.015, 0.05],
    "n_estimators": [300, 450],
    "half_life_days": [140, 180, 220],
}

best_auc = float("-inf")
best_params = None

for max_depth, eta, n_estimators, half_life_days in product(
    search_space["max_depth"],
    search_space["eta"],
    search_space["n_estimators"],
    search_space["half_life_days"],
):
    candidate = {
        **configured,
        "max_depth": max_depth,
        "eta": eta,
        "n_estimators": n_estimators,
    }
    model = xgb.XGBClassifier(**{**candidate, "objective": "binary:logistic",
            "eval_metric": "logloss",})
    model.fit(
        clf_train[features],
        y_train,
        sample_weight=_recency_weights(clf_train["as_of_date"], half_life_days),
    )
    auc = roc_auc_score(y_valid, model.predict_proba(clf_valid[features])[:, 1])

    if auc > best_auc:
        best_auc = auc
        best_params = candidate
        best_half_life = half_life_days

merged_xgb_params = best_params
half_life_days = best_half_life

assert merged_xgb_params is not None and half_life_days > 0
print(f"Best validation ROC AUC: {best_auc:.3f}")
print(f"Selected half-life: {half_life_days} days")
print(f"Selected XGBoost parameters: {merged_xgb_params}")

Best validation ROC AUC: 0.789
Selected half-life: 180 days
Selected XGBoost parameters: {'eta': 0.005, 'max_depth': 3, 'subsample': 0.9, 'colsample_bytree': 0.9, 'n_estimators': 300, 'random_state': 42}


In [21]:
# TODO cant put halflife in this function as of now

before = walk_forward_evaluate(historical, xgb_params=params['model']['xgboost'])
before.mean()[["auc", "accuracy", "baseline_accuracy"]]

auc                  0.670006
accuracy             0.640639
baseline_accuracy    0.665677
dtype: float64

In [22]:
after = walk_forward_evaluate(historical, xgb_params=merged_xgb_params)
before.mean()[["auc", "accuracy", "baseline_accuracy"]]
after.mean()[["auc", "accuracy", "baseline_accuracy"]]

auc                  0.670006
accuracy             0.640639
baseline_accuracy    0.665677
dtype: float64

In [23]:
if NORI_AVAILABLE:
    # Place the code that should run when NORI is available here
    # currently can only run in dev container cus computer too old
    # but computer also too old to run dev container it seems
    # but id like to compare it to my tuned XGBoost model
    pass